# TourGuide TTS/Translation Microservice

Ce notebook lance le microservice FastAPI sur Google Colab avec un GPU T4 gratuit.
Un tunnel ngrok expose le service pour que le Studio Web puisse l'appeler.

## Prérequis
1. **Runtime GPU** : Menu → Runtime → Change runtime type → **T4 GPU**
2. **Compte ngrok gratuit** : https://ngrok.com → récupère ton authtoken

## Étapes
1. Exécute chaque cellule dans l'ordre
2. Copie l'URL ngrok affichée
3. Colle-la dans ton `.env.local` : `NEXT_PUBLIC_MICROSERVICE_URL=https://xxxx.ngrok-free.app`
4. Le service tourne tant que le notebook est ouvert (~12h max)

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print("\u2705 GPU pr\u00eat")
else:
    print("\u274c Pas de GPU ! Va dans Runtime \u2192 Change runtime type \u2192 T4 GPU")

## 2. Installer les dépendances

In [ ]:
!pip install -q fastapi uvicorn qwen-tts soundfile pydub transformers sentencepiece pyngrok requests
!pip install -U flash-attn --no-build-isolation -q 2>/dev/null || echo "flash-attn optionnel"
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("\u2705 D\u00e9pendances install\u00e9es")

## 3. Configurer ngrok

Va sur https://dashboard.ngrok.com/get-started/your-authtoken et copie ton token.

In [ ]:
NGROK_AUTHTOKEN = ""  # @param {type:"string"}
MICROSERVICE_API_KEY = "tourguide-tts-2026"  # @param {type:"string"}

if not NGROK_AUTHTOKEN:
    print("\u274c Colle ton ngrok authtoken ci-dessus !")
    print("   https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    import os
    os.environ["MICROSERVICE_API_KEY"] = MICROSERVICE_API_KEY
    print(f"\u2705 API Key configur\u00e9e : {MICROSERVICE_API_KEY[:10]}...")
    print(f"\u2705 ngrok token configur\u00e9")

## 4. Charger les modèles TTS + Traduction

⏳ Première exécution : ~3-5 min (téléchargement modèles).
Les exécutions suivantes utilisent le cache.

In [ ]:
import torch
import logging
import time
import base64
import io
import re
import os
import tempfile
from urllib.parse import urlparse

import soundfile as sf
import requests as req_lib

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("tourguide")

# ── TTS : Qwen3-TTS 0.6B ──
print("\u23f3 Chargement de Qwen3-TTS 0.6B...")
start = time.time()

from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
tts_processor = AutoProcessor.from_pretrained(MODEL_NAME)
tts_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
tts_ready = True
print(f"\u2705 TTS charg\u00e9 en {time.time()-start:.0f}s")

# ── Traduction : MarianMT (lazy) ──
from transformers import MarianMTModel, MarianTokenizer

MARIAN_MODELS = {
    ("fr", "en"): "Helsinki-NLP/opus-mt-fr-en",
    ("fr", "it"): "Helsinki-NLP/opus-mt-fr-it",
    ("fr", "de"): "Helsinki-NLP/opus-mt-fr-de",
    ("fr", "es"): "Helsinki-NLP/opus-mt-fr-es",
}

translation_models = {}
translation_tokenizers = {}

def load_translation_pair(src, tgt):
    pair = (src, tgt)
    if pair in translation_models:
        return True
    model_name = MARIAN_MODELS.get(pair)
    if not model_name:
        return False
    print(f"\u23f3 Chargement MarianMT {src}\u2192{tgt}...")
    translation_tokenizers[pair] = MarianTokenizer.from_pretrained(model_name)
    translation_models[pair] = MarianMTModel.from_pretrained(model_name)
    print(f"\u2705 MarianMT {src}\u2192{tgt} charg\u00e9")
    return True

print(f"\n\u2705 Microservice pr\u00eat (TTS + traduction lazy)")
print(f"   VRAM utilis\u00e9e : {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 5. Définir l'API FastAPI

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

app = FastAPI(title="TourGuide TTS Microservice (Colab)")

# CORS pour le Studio Web
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

API_KEY = os.environ.get("MICROSERVICE_API_KEY", "")

@app.middleware("http")
async def verify_api_key(request: Request, call_next):
    if request.url.path == "/health":
        return await call_next(request)
    if API_KEY and request.headers.get("X-API-Key") != API_KEY:
        return JSONResponse(status_code=401, content={"detail": "Invalid API key"})
    return await call_next(request)

# ── Models ──
class TTSRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=10000)
    language: str = Field(default="fr", pattern="^(fr|en|it|de|es|ja|ko|zh|ru)$")
    voice_id: str | None = None

class TranslateRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=50000)
    source_lang: str = Field(default="fr", pattern="^(fr|en|it|de|es)$")
    target_lang: str = Field(..., pattern="^(fr|en|it|de|es)$")

class SilenceRequest(BaseModel):
    audio_url: str = Field(..., min_length=1)

# ── Endpoints ──
@app.get("/health")
async def health():
    return {
        "status": "ok",
        "tts": tts_ready,
        "translation": True,
        "silence_detection": True,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_used_gb": round(torch.cuda.memory_allocated()/1e9, 1),
    }

DEFAULT_VOICES = {"fr": "Chelsie", "en": "Chelsie", "it": "Chelsie", "de": "Chelsie", "es": "Chelsie"}

@app.post("/v1/tts/generate")
async def generate_tts(req: TTSRequest):
    if not tts_ready:
        return {"ok": False, "error": "TTS not ready"}
    try:
        voice = req.voice_id or DEFAULT_VOICES.get(req.language, "Chelsie")
        sanitized = re.sub(r"<\|[^|]*\|>", "", req.text)
        prompt = f"<|speaker:{voice}|><|lang:{req.language}|>{sanitized}"

        inputs = tts_processor(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = tts_model.generate(**inputs, max_new_tokens=4096, do_sample=True, temperature=0.7)
        audio_array = tts_processor.decode_audio(outputs[0])
        torch.cuda.empty_cache()

        sample_rate = 24000
        buf = io.BytesIO()
        sf.write(buf, audio_array, sample_rate, format="WAV")
        buf.seek(0)
        audio_b64 = base64.b64encode(buf.read()).decode("ascii")
        duration_ms = int(len(audio_array) / sample_rate * 1000)

        return {"ok": True, "audio_base64": audio_b64, "duration_ms": duration_ms}
    except Exception as e:
        torch.cuda.empty_cache()
        logger.error(f"TTS error: {e}")
        return {"ok": False, "error": str(e)}

@app.post("/v1/translate/marianmt")
async def translate(req: TranslateRequest):
    if req.source_lang == req.target_lang:
        return {"ok": True, "translated_text": req.text}
    pair = (req.source_lang, req.target_lang)
    if not load_translation_pair(*pair):
        return {"ok": False, "error": f"Paire non support\u00e9e: {pair}"}
    try:
        tokenizer = translation_tokenizers[pair]
        model = translation_models[pair]
        inputs = tokenizer(req.text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            translated = model.generate(**inputs)
        result = tokenizer.decode(translated[0], skip_special_tokens=True)
        return {"ok": True, "translated_text": result}
    except Exception as e:
        logger.error(f"Translation error: {e}")
        return {"ok": False, "error": str(e)}

ALLOWED_HOSTS = {"s3.amazonaws.com", "s3.us-east-1.amazonaws.com"}
def is_allowed_url(url):
    try:
        parsed = urlparse(url)
        host = parsed.hostname or ""
        return host in ALLOWED_HOSTS or (host.endswith(".amazonaws.com") and ".s3." in host)
    except:
        return False

@app.post("/v1/silence-detect")
async def silence_detect(req: SilenceRequest):
    if not is_allowed_url(req.audio_url):
        return {"ok": False, "error": "URL non autoris\u00e9e"}
    try:
        from pydub import AudioSegment
        from pydub.silence import detect_nonsilent
        resp = req_lib.get(req.audio_url, timeout=30, allow_redirects=False)
        resp.raise_for_status()
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp.write(resp.content)
            tmp_path = tmp.name
        try:
            audio = AudioSegment.from_file(tmp_path)
            segments = detect_nonsilent(audio, min_silence_len=800, silence_thresh=-40)
            if not segments:
                segments = [(0, len(audio))]
            return {"ok": True, "segments": [{"start_ms": s[0], "end_ms": s[1]} for s in segments]}
        finally:
            os.unlink(tmp_path)
    except Exception as e:
        logger.error(f"Silence detection error: {e}")
        return {"ok": False, "error": str(e)}

print("\u2705 API FastAPI d\u00e9finie")

## 6. Lancer le serveur + tunnel ngrok

Cette cellule tourne en continu. Le serveur reste actif tant que le notebook est ouvert.

**Copie l'URL ngrok** affichée et colle-la dans ton `.env.local` :
```
NEXT_PUBLIC_MICROSERVICE_URL=https://xxxx-xx-xx-xxx-xxx.ngrok-free.app
NEXT_PUBLIC_MICROSERVICE_API_KEY=tourguide-tts-2026
```

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok, conf

# Configurer ngrok
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Ouvrir le tunnel
public_url = ngrok.connect(8000, "http")

print("=" * 60)
print(f"\u2705 MICROSERVICE EN LIGNE")
print(f"")
print(f"   URL publique : {public_url}")
print(f"   API Key      : {API_KEY}")
print(f"")
print(f"   Ajoute dans ton .env.local :")
print(f"   NEXT_PUBLIC_MICROSERVICE_URL={public_url}")
print(f"   NEXT_PUBLIC_MICROSERVICE_API_KEY={API_KEY}")
print(f"")
print(f"   Test : curl {public_url}/health")
print("=" * 60)

import uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)

## 7. Test rapide (optionnel)

Ouvre un autre onglet Colab ou utilise curl pour tester :

In [ ]:
# \u00c0 ex\u00e9cuter dans un AUTRE notebook ou terminal :
#
# curl -X POST https://VOTRE-URL.ngrok-free.app/v1/tts/generate \
#   -H "Content-Type: application/json" \
#   -H "X-API-Key: tourguide-tts-2026" \
#   -d '{"text": "Bienvenue sur la C\u00f4te d\u0027Azur.", "language": "fr"}'
#
# curl -X POST https://VOTRE-URL.ngrok-free.app/v1/translate/marianmt \
#   -H "Content-Type: application/json" \
#   -H "X-API-Key: tourguide-tts-2026" \
#   -d '{"text": "Bienvenue \u00e0 Nice.", "source_lang": "fr", "target_lang": "en"}'

print("Voir les commandes curl ci-dessus pour tester depuis un autre terminal.")